## Depression Detection — Participant-Level Pipeline (E-DAIC, OpenFace AU Features, BiLSTM + Masked Attention Pooling)

Description: This notebook implements a participant-level pipeline for depression detection using the E-DAIC dataset. It leverages OpenFace Action Unit (AU) features extracted from video data. The core methodology involves:

Preprocessing: Loading raw OpenFace features, filtering for high-confidence frames, and dropping irrelevant features.

Windowing: Segmenting the preprocessed data into fixed-size windows with an overlap.

Data Splitting & Metadata Generation: Creating train, validation, and test sets at the participant level and generating metadata for efficient data loading.

Feature Scaling: Fitting a StandardScaler on the training data only to prevent data leakage.

Custom Dataset and DataLoader: Implementing a ParticipantDataset to handle variable-length participant data and participant_collate for batching with padding.

Model Architecture: A ParticipantDepressionNet with a two-level architecture: a BiLSTM with attention pooling for per-window embeddings, followed by masked mean pooling across valid windows to obtain a participant-level embedding. This embedding is then passed to a shared layer for both binary classification (depression/non-depression) and regression (PHQ-8 score prediction).

Training & Evaluation: The model is trained using a combined classification and regression loss with class weights for imbalance. It includes early stopping based on validation F1-score and evaluates performance on a held-out test set using metrics like accuracy, precision, recall, F1-score, MAE, RMSE, and R2.

## 1. Mount Drive & Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import joblib
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix,
    mean_absolute_error, mean_squared_error, r2_score
)

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

## 2. Paths & Config

In [ ]:
BASE_DIR      = Path("/content/drive/MyDrive")
PROCESSED_DIR = BASE_DIR / "DAIC_WOZ_VIDEO_BRANCH" / "data" / "processed"
WINDOWS_DIR   = PROCESSED_DIR / "windows_processed"
NEW_PROC_DIR  = PROCESSED_DIR / "new_processed"

FPS         = 30
WINDOW_SIZE = 300    # 10 sec @ 30fps
STRIDE      = 150    # 5 sec overlap
BATCH_SIZE  = 4      # participants per batch
NUM_EPOCHS  = 80
PATIENCE    = 20
LR          = 1e-3
ALPHA       = 0.05   # regression loss weight
DROP_FEATURES = ["frame", "timestamp", "confidence", "success"]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"PROCESSED_DIR exists: {PROCESSED_DIR.exists()}")

Device: cuda
PROCESSED_DIR exists: True


## 3. Preprocessing

In [ ]:
def load_participant(path):
    df = pd.read_csv(path)
    print(f"Loaded: {path}")
    return df

def filter_frame(df):
    df_filtered = df[(df["success"] == 1) & (df["confidence"] >= 0.8)]
    original = len(df)
    filtered = len(df_filtered)
    print(f"Original: {original} | Filtered: {filtered} | Removed: {(original-filtered)/original:.2%}")
    return df_filtered

def select_feature(df, drop_features):
    return df.drop(columns=drop_features)

def save_processed_file(df, pid):
    df.to_csv(NEW_PROC_DIR / f"{pid}_nprocessed.csv", index=False)

def preprocessing_driver_script(session_ids, drop_features):
    summary = []
    for pid in session_ids:
        try:
            print(f"\nProcessing Participant {pid}...")
            file_path = (
                BASE_DIR / "edaic3.0" / str(pid) / "features"
                / f"{pid}_OpenFace2.1.0_Pose_gaze_AUs.csv"
            )
            df = load_participant(file_path)
            original_frames = len(df)
            df_filtered = filter_frame(df)
            filtered_frames = len(df_filtered)
            removal_ratio = (original_frames - filtered_frames) / original_frames
            df_filtered = select_feature(df_filtered, drop_features)
            save_processed_file(df_filtered, pid)
            summary.append({
                "participant_id":  pid,
                "original_frames": original_frames,
                "filtered_frames": filtered_frames,
                "removal_ratio":   removal_ratio,
                "num_features":    df_filtered.shape[1]
            })
            print(f"Saved | Frames: {original_frames} -> {filtered_frames}")
        except Exception as e:
            print(f"Error processing {pid}: {e}")
    summary_df = pd.DataFrame(summary)
    summary_df.to_csv(NEW_PROC_DIR / "preprocessing_summary.csv", index=False)
    return summary_df

label_df    = pd.read_csv(BASE_DIR / "edaic3.0" / "labels" / "Detailed_PHQ8_Labels.csv")
session_ids = label_df["Participant_ID"].tolist()

summary_df = preprocessing_driver_script(session_ids, DROP_FEATURES)
summary_df.head()


Processing Participant 300...
Loaded: /content/drive/MyDrive/edaic3.0/300/features/300_OpenFace2.1.0_Pose_gaze_AUs.csv
Original: 19458 | Filtered: 19378 | Removed: 0.41%
Saved | Frames: 19458 -> 19378

Processing Participant 301...
Loaded: /content/drive/MyDrive/edaic3.0/301/features/301_OpenFace2.1.0_Pose_gaze_AUs.csv
Original: 24721 | Filtered: 24618 | Removed: 0.42%
Saved | Frames: 24721 -> 24618

Processing Participant 302...
Loaded: /content/drive/MyDrive/edaic3.0/302/features/302_OpenFace2.1.0_Pose_gaze_AUs.csv
Original: 22766 | Filtered: 22130 | Removed: 2.79%
Saved | Frames: 22766 -> 22130

Processing Participant 303...


KeyboardInterrupt: 

In [ ]:
label_df    = pd.read_csv(BASE_DIR / "edaic3.0" / "labels" / "Detailed_PHQ8_Labels.csv")
session_ids = label_df["Participant_ID"].tolist()

# summary_df = preprocessing_driver_script(session_ids, DROP_FEATURES)
# summary_df.head()

## 4. Windowing

In [ ]:
def generate_windows(df, window_size=WINDOW_SIZE, stride=STRIDE):
    data = df.values
    windows = []
    start = 0
    while start + window_size <= len(data):
        windows.append(data[start:start + window_size])
        start += stride
    return np.array(windows)

def save_windows(windows, pid):
    np.save(WINDOWS_DIR / f"{pid}_windows.npy", windows)

for pid in session_ids:
    df = pd.read_csv(NEW_PROC_DIR / f"{pid}_nprocessed.csv")
    windows = generate_windows(df)
    save_windows(windows, pid)
    print(f"{pid}: {windows.shape[0]} windows")

300: 128 windows
301: 163 windows
302: 146 windows
303: 195 windows
304: 154 windows
305: 338 windows
306: 169 windows
307: 244 windows
308: 167 windows
309: 140 windows
310: 167 windows
311: 155 windows
312: 157 windows
313: 149 windows
314: 309 windows
315: 192 windows
316: 173 windows
317: 161 windows
318: 116 windows
319: 132 windows
320: 166 windows
321: 162 windows
322: 207 windows
323: 161 windows
324: 141 windows
325: 173 windows
326: 136 windows
327: 134 windows
328: 210 windows
329: 138 windows
330: 151 windows
331: 167 windows
332: 173 windows
333: 192 windows
334: 193 windows
335: 163 windows
336: 185 windows
337: 372 windows
338: 114 windows
339: 170 windows
340: 118 windows
341: 172 windows
343: 175 windows
344: 215 windows
345: 156 windows
346: 242 windows
347: 119 windows
348: 142 windows
349: 239 windows
350: 174 windows
351: 151 windows
352: 150 windows
353: 155 windows
354: 113 windows
355: 131 windows
356: 187 windows
357: 78 windows
358: 121 windows
359: 199 window

## 5. Split & Metadata

In [ ]:
CSV_PATH = "/content/drive/MyDrive/DAIC_WOZ_AUDIO_BRANCH/data/processed/wav2vec_5utt_segments_participant_split.csv"
df_split = pd.read_csv(CSV_PATH)
df_split = df_split.drop(columns=[
    "sample_id", "sample_path", "num_samples",
    "duration_sec", "binary_label", "PHQ8_Score", "num_segments_merged"
])
df_split.loc[438] = [438, "train"]
df_unique = df_split.drop_duplicates(subset=["Participant_ID"], keep="first")

label_df = pd.read_csv(BASE_DIR / "edaic3.0" / "labels" / "Detailed_PHQ8_Labels.csv")
label_df = pd.merge(df_unique, label_df, on="Participant_ID", how="inner")
print("Merged label_df shape:", label_df.shape)

Merged label_df shape: (219, 13)


In [ ]:
def build_metadata(label_df, windows_dir):
    metadata = []
    for _, row in label_df.iterrows():
        pid = row["Participant_ID"]
        window_file = windows_dir / f"{pid}_windows.npy"
        if not window_file.exists():
            print(f"Missing windows for {pid}")
            continue
        windows = np.load(window_file)
        metadata.append({
            "participant_id": pid,
            "phq_score":      row["PHQ_8Total"],
            "binary_label":   row["PHQ8_Binary"],
            "num_windows":    windows.shape[0],
            "window_file":    str(window_file),
            "split":          row["split"]
        })
    return pd.DataFrame(metadata)

metadata_df = build_metadata(label_df, WINDOWS_DIR)
metadata_df.to_csv(WINDOWS_DIR / "metadata.csv", index=False)
metadata_df.head()

KeyboardInterrupt: 

In [ ]:
metadata_df = pd.read_csv(WINDOWS_DIR / "metadata.csv")

train_df = metadata_df[metadata_df["split"] == "train"].reset_index(drop=True)
val_df   = metadata_df[metadata_df["split"] == "val"].reset_index(drop=True)
test_df  = metadata_df[metadata_df["split"] == "test"].reset_index(drop=True)

print(f"Train: {len(train_df)} participants | Val: {len(val_df)} | Test: {len(test_df)}")
print("\nTrain labels:\n", train_df["binary_label"].value_counts())
print("\nVal labels:\n",   val_df["binary_label"].value_counts())
print("\nTest labels:\n",  test_df["binary_label"].value_counts())

Train: 131 participants | Val: 44 | Test: 44

Train labels:
 binary_label
0    92
1    39
Name: count, dtype: int64

Val labels:
 binary_label
0    31
1    13
Name: count, dtype: int64

Test labels:
 binary_label
0    31
1    13
Name: count, dtype: int64


## 6. Fit Scaler on Train Participants Only
> Scaler is fit only on train frames to prevent data leakage.

In [ ]:
all_train_frames = []
for _, row in train_df.iterrows():
    raw    = np.load(row["window_file"])             # (W, 300, 49)
    frames = raw.reshape(-1, raw.shape[-1])           # (W*300, 49)
    all_train_frames.append(frames)

all_train_frames = np.concatenate(all_train_frames, axis=0)
print("Train frames shape:", all_train_frames.shape)

scaler = StandardScaler()
scaler.fit(all_train_frames)
joblib.dump(scaler, PROCESSED_DIR / "scaler.pkl")
print("Scaler saved to:", PROCESSED_DIR / "scaler.pkl")

# Sanity check
sample = scaler.transform(all_train_frames[:1000])
print("Mean (should be ~0):", sample.mean(axis=0)[:5].round(4))
print("Std  (should be ~1):", sample.std(axis=0)[:5].round(4))

Train frames shape: (7309500, 49)
Scaler saved to: /content/drive/MyDrive/DAIC_WOZ_VIDEO_BRANCH/data/processed/scaler.pkl
Mean (should be ~0): [ 0.3119 -0.4648  0.595  -0.2862 -1.0855]
Std  (should be ~1): [0.7563 0.6216 0.3346 1.2128 1.4556]


## 7. Participant-Level Dataset & DataLoaders

In [ ]:
class ParticipantDataset(Dataset):
    """
    One sample = one participant (all their windows).
    Scaler is applied here — val/test are transformed with train scaler,
    which is the correct way to avoid leakage.
    """
    def __init__(self, metadata_df, scaler):
        self.metadata = metadata_df.reset_index(drop=True)
        self.scaler   = scaler

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row      = self.metadata.iloc[idx]
        raw      = np.load(row["window_file"]).astype(np.float32)  # (W, 300, 49)
        W, T, F  = raw.shape
        scaled   = self.scaler.transform(
            raw.reshape(-1, F)
        ).reshape(W, T, F).astype(np.float32)
        return {
            "windows":        torch.from_numpy(scaled),
            "binary_label":   torch.tensor(row["binary_label"], dtype=torch.long),
            "phq_score":      torch.tensor(row["phq_score"],    dtype=torch.float32),
            "participant_id": row["participant_id"]
        }


def participant_collate(batch):
    """
    Pads windows along the W (num_windows) dimension so variable-length
    participants can be batched. num_windows tracks the real lengths
    so padding is masked out in the model.
    """
    windows     = [x["windows"] for x in batch]                # list of (W_i, 300, 49)
    num_windows = torch.tensor([w.shape[0] for w in windows])  # (B,)
    padded      = pad_sequence(windows, batch_first=True)       # (B, W_max, 300, 49)
    return {
        "windows":        padded,
        "num_windows":    num_windows,
        "binary_label":   torch.stack([x["binary_label"] for x in batch]),
        "phq_score":      torch.stack([x["phq_score"]    for x in batch]),
        "participant_id": [x["participant_id"]            for x in batch]
    }


scaler = joblib.load(PROCESSED_DIR / "scaler.pkl")

train_dataset = ParticipantDataset(train_df, scaler)
val_dataset   = ParticipantDataset(val_df,   scaler)
test_dataset  = ParticipantDataset(test_df,  scaler)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=participant_collate
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=participant_collate
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=participant_collate
)

batch = next(iter(train_loader))
print("windows shape: ", batch["windows"].shape)     # (B, W_max, 300, 49)
print("num_windows:   ", batch["num_windows"])
print("binary_label:  ", batch["binary_label"])

windows shape:  torch.Size([4, 286, 300, 49])
num_windows:    tensor([272, 286, 116, 235])
binary_label:   tensor([0, 0, 0, 0])


## 8. Model

In [ ]:
class ParticipantDepressionNet(nn.Module):
    """
    Two-level architecture:
      Level 1 — BiLSTM + attention pools each 10-sec window -> window embedding
      Level 2 — Masked mean across VALID windows only -> participant embedding

    Key fixes vs original:
      - num_windows mask excludes zero-padded windows from the mean
      - LayerNorm after shared linear for stable training
      - 2-layer BiLSTM with dropout
    """
    def __init__(
        self,
        input_size=49,
        hidden_size=64,
        num_layers=2,
        dropout=0.3
    ):
        super().__init__()

        # --- Level 1: per-window encoder ---
        self.window_encoder = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )
        lstm_out_dim = hidden_size * 2  # 128
        self.window_attention = nn.Linear(lstm_out_dim, 1)

        # --- Level 2: participant classifier/regressor ---
        self.shared = nn.Sequential(
            nn.Linear(lstm_out_dim, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.classifier = nn.Linear(64, 2)
        self.regressor  = nn.Linear(64, 1)

    def encode_window(self, windows_flat):
        """windows_flat: (B*W, T, F) -> (B*W, lstm_out_dim)"""
        lstm_out, _ = self.window_encoder(windows_flat)          # (B*W, T, 128)
        scores      = self.window_attention(lstm_out).squeeze(-1) # (B*W, T)
        weights     = torch.softmax(scores, dim=1)                # (B*W, T)
        embeddings  = (lstm_out * weights.unsqueeze(-1)).sum(dim=1) # (B*W, 128)
        return embeddings

    def forward(self, windows, num_windows):
        """
        windows:     (B, W_max, T, F)
        num_windows: (B,)  — real window count per participant
        """
        B, W, T, F   = windows.shape
        windows_flat = windows.reshape(B * W, T, F)
        embeddings   = self.encode_window(windows_flat)  # (B*W, 128)
        embeddings   = embeddings.reshape(B, W, -1)      # (B, W, 128)

        # Mask out padding before averaging
        mask = torch.zeros(B, W, 1, device=windows.device)
        for i, n in enumerate(num_windows):
            mask[i, :n] = 1.0

        participant_emb = (
            (embeddings * mask).sum(dim=1)
            / mask.sum(dim=1).clamp(min=1)
        )  # (B, 128)

        shared = self.shared(participant_emb)
        return {
            "class_logits": self.classifier(shared),  # (B, 2)
            "phq_score":    self.regressor(shared)    # (B, 1)
        }


# Shape test
model = ParticipantDepressionNet().to(device)
out   = model(
    batch["windows"].to(device),
    batch["num_windows"]
)
print("class_logits:", out["class_logits"].shape)
print("phq_score:   ", out["phq_score"].shape)

class_logits: torch.Size([4, 2])
phq_score:    torch.Size([4, 1])


## 9. Loss, Optimizer, Scheduler

In [ ]:
# Class weights computed from train participants only
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["binary_label"].values
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("Class weights:", class_weights)

cls_criterion = nn.CrossEntropyLoss(weight=class_weights)
reg_criterion = nn.MSELoss()

model     = ParticipantDepressionNet().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

Class weights: tensor([0.7120, 1.6795])


## 10. Train & Validate Functions

In [ ]:
def train_one_epoch(model, loader, optimizer, device, alpha=ALPHA):
    model.train()
    total_loss = total_cls = total_reg = 0

    for batch in loader:
        windows       = batch["windows"].to(device)
        num_windows   = batch["num_windows"]
        binary_labels = batch["binary_label"].to(device)
        phq_scores    = batch["phq_score"].to(device)

        optimizer.zero_grad()
        outputs  = model(windows, num_windows)

        cls_loss = cls_criterion(outputs["class_logits"], binary_labels)
        reg_loss = reg_criterion(outputs["phq_score"].squeeze(1), phq_scores)
        loss     = cls_loss + alpha * reg_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_cls  += cls_loss.item()
        total_reg  += reg_loss.item()

    n = len(loader)
    return total_loss / n, total_cls / n, total_reg / n


def validate(model, loader, device, alpha=ALPHA):
    model.eval()
    total_loss   = 0
    all_preds    = []; all_labels   = []
    all_phq_pred = []; all_phq_true = []

    with torch.no_grad():
        for batch in loader:
            windows       = batch["windows"].to(device)
            num_windows   = batch["num_windows"]
            binary_labels = batch["binary_label"].to(device)
            phq_scores    = batch["phq_score"].to(device)

            outputs  = model(windows, num_windows)
            cls_loss = cls_criterion(outputs["class_logits"], binary_labels)
            reg_loss = reg_criterion(outputs["phq_score"].squeeze(1), phq_scores)
            total_loss += (cls_loss + alpha * reg_loss).item()

            preds = torch.argmax(outputs["class_logits"], dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(binary_labels.cpu().numpy())
            all_phq_pred.extend(outputs["phq_score"].squeeze(1).cpu().numpy())
            all_phq_true.extend(phq_scores.cpu().numpy())

    f1   = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    acc  = accuracy_score(all_labels, all_preds)
    rmse = np.sqrt(mean_squared_error(all_phq_true, all_phq_pred))
    return total_loss / len(loader), acc, f1, rmse

## 11. Training Loop

In [ ]:
best_f1          = 0
patience_counter = 0
history          = []

print("=" * 65)
print(f"Device: {device}  |  Train: {len(train_dataset)} participants  |  Val: {len(val_dataset)}")
print(f"Batch: {BATCH_SIZE}  |  LR: {LR}  |  Epochs: {NUM_EPOCHS}  |  Patience: {PATIENCE}")
print("=" * 65)

for epoch in range(NUM_EPOCHS):

    tr_loss, tr_cls, tr_reg = train_one_epoch(
        model, train_loader, optimizer, device
    )
    val_loss, val_acc, val_f1, val_rmse = validate(
        model, val_loader, device
    )
    scheduler.step()

    history.append({
        "epoch":      epoch + 1,
        "train_loss": tr_loss,
        "val_loss":   val_loss,
        "val_acc":    val_acc,
        "val_f1":     val_f1,
        "val_rmse":   val_rmse
    })

    print(
        f"Epoch {epoch+1:03d} | "
        f"TrLoss={tr_loss:.4f} (cls={tr_cls:.4f} reg={tr_reg:.4f}) | "
        f"ValLoss={val_loss:.4f} | "
        f"Acc={val_acc:.4f} | "
        f"F1(macro)={val_f1:.4f} | "
        f"RMSE={val_rmse:.4f}"
    )

    if val_f1 > best_f1:
        best_f1 = val_f1
        patience_counter = 0
        torch.save(model.state_dict(), "best_participant_model.pt")
        print(f"  New best F1={best_f1:.4f} — model saved")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

print(f"\nBest val macro-F1: {best_f1:.4f}")

Device: cpu  |  Train: 131 participants  |  Val: 44
Batch: 4  |  LR: 0.001  |  Epochs: 80  |  Patience: 20


## 12. Training Curves

In [ ]:
hist = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(hist["epoch"], hist["train_loss"], label="Train")
axes[0].plot(hist["epoch"], hist["val_loss"],   label="Val")
axes[0].set_title("Loss"); axes[0].legend()

axes[1].plot(hist["epoch"], hist["val_f1"])
axes[1].set_title("Val Macro-F1")

axes[2].plot(hist["epoch"], hist["val_rmse"])
axes[2].set_title("Val RMSE")

plt.tight_layout()
plt.savefig("training_curves.png", dpi=120)
plt.show()

## 13. Evaluate on Test Set

In [ ]:
def evaluate(model, loader, device):
    model.eval()
    all_cls_preds  = []; all_cls_labels = []
    all_phq_preds  = []; all_phq_labels = []

    with torch.no_grad():
        for batch in loader:
            windows       = batch["windows"].to(device)
            num_windows   = batch["num_windows"]
            binary_labels = batch["binary_label"].to(device)
            phq_scores    = batch["phq_score"].to(device)

            outputs   = model(windows, num_windows)
            cls_preds = torch.argmax(outputs["class_logits"], dim=1)

            all_cls_preds.extend(cls_preds.cpu().numpy())
            all_cls_labels.extend(binary_labels.cpu().numpy())
            all_phq_preds.extend(outputs["phq_score"].squeeze(1).cpu().numpy())
            all_phq_labels.extend(phq_scores.cpu().numpy())

    return {
        "accuracy":         accuracy_score(all_cls_labels, all_cls_preds),
        "precision":        precision_score(all_cls_labels, all_cls_preds, average="macro", zero_division=0),
        "recall":           recall_score(all_cls_labels, all_cls_preds,    average="macro", zero_division=0),
        "f1":               f1_score(all_cls_labels, all_cls_preds,        average="macro", zero_division=0),
        "confusion_matrix": confusion_matrix(all_cls_labels, all_cls_preds),
        "mae":              mean_absolute_error(all_phq_labels, all_phq_preds),
        "rmse":             np.sqrt(mean_squared_error(all_phq_labels, all_phq_preds)),
        "r2":               r2_score(all_phq_labels, all_phq_preds)
    }


model.load_state_dict(torch.load("best_participant_model.pt", map_location=device))
model.to(device)

test_metrics = evaluate(model, test_loader, device)

print("\n" + "=" * 50)
print("TEST RESULTS  (participant-level)")
print("=" * 50)
for k, v in test_metrics.items():
    if k != "confusion_matrix":
        print(f"{k:<15}: {v:.4f}")
print("\nConfusion Matrix:")
print(test_metrics["confusion_matrix"])